# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd
import numpy as np
from scipy import stats

In [2]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')
print('orders:', orders.shape)
print('catalog:', catalog.shape)
print('marketing:', marketing.shape)

orders: (25100, 12)
catalog: (7, 4)
marketing: (1620, 5)


In [3]:
# explorar datasets: orders
print('=== INFO GENERAL ===')
orders.info()

print('\n=== PRIMERAS FILAS ===')
display(orders.head())

print('\n=== VALORES NULOS POR COLUMNA ===')
print(orders.isna().sum())

print('\n=== DUPLICADOS ===')
print('Filas completamente duplicadas:', orders.duplicated().sum())
print('id_pedido repetidos:', orders['id_pedido'].duplicated().sum())

print('\n=== ESTADÍSTICAS DE COLUMNAS NUMÉRICAS ===')
display(orders.describe())


=== INFO GENERAL ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB

=== PRIMERAS FILAS ===


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28



=== VALORES NULOS POR COLUMNA ===
id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     80
cantidad               50
precio_unitario        50
monto_descuento        50
monto_total             0
dtype: int64

=== DUPLICADOS ===
Filas completamente duplicadas: 100
id_pedido repetidos: 100

=== ESTADÍSTICAS DE COLUMNAS NUMÉRICAS ===


,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [4]:
# explorar datasets: catalog
print('=== INFO GENERAL ===')
catalog.info()

print('\n=== TABLA COMPLETA ===')
display(catalog)

print('\n=== DUPLICADOS ===')
print('Filas duplicadas:', catalog.duplicated().sum())
print('Productos repetidos:', catalog['nombre_producto'].duplicated().sum())

# (extra) comparar categorías entre catalog y orders
print('\n=== CATEGORÍAS: catalog vs orders ===')
print('catalog:', sorted(catalog['categoria_producto'].unique()))
print('orders :', sorted(orders['categoria_producto'].dropna().unique()))

=== INFO GENERAL ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes

=== TABLA COMPLETA ===


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"
5,Sneakers-Urban-42,Moda,17.21,Greene-Smith
6,Jacket-Winter-M,Moda,189.31,Mcmillan-Rhodes



=== DUPLICADOS ===
Filas duplicadas: 0
Productos repetidos: 0

=== CATEGORÍAS: catalog vs orders ===
catalog: ['Electrónica', 'Hogar', 'Moda']
orders : ['Electronica', 'Hogar', 'Moda']


In [5]:
# explorar datasets: marketing
print('=== INFO GENERAL ===')
marketing.info()

print('\n=== PRIMERAS FILAS ===')
display(marketing.head())

print('\n=== VALORES NULOS POR COLUMNA ===')
print(marketing.isna().sum())

print('\n=== DUPLICADOS ===')
print('Filas duplicadas:', marketing.duplicated().sum())

print('\n=== ESTADÍSTICAS DE GASTO ===')
display(marketing['gasto'].describe())

print('\n=== RANGO DE FECHAS ===')
print('Desde:', marketing['fecha'].min(), '| Hasta:', marketing['fecha'].max())

# (extra) valores únicos de las columnas categóricas
print('\n=== VALORES ÚNICOS ===')
for col in ['pais', 'canal']:
    print(f'{col}:', marketing[col].unique())

=== INFO GENERAL ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB

=== PRIMERAS FILAS ===


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40



=== VALORES NULOS POR COLUMNA ===
fecha           0
pais            0
id_campaña      0
canal         101
gasto           0
dtype: int64

=== DUPLICADOS ===
Filas duplicadas: 0

=== ESTADÍSTICAS DE GASTO ===


count    1620.00000
mean     1772.74292
std       734.43294
min       501.11000
25%      1128.03000
50%      1782.42500
75%      2420.68500
max      2999.36000
Name: gasto, dtype: float64


=== RANGO DE FECHAS ===
Desde: 2025-01-01 | Hasta: 2025-06-29

=== VALORES ÚNICOS ===
pais: ['Mexico' 'Colombia' 'Argentina']
canal: ['organic' 'paid_search' 'social' nan]


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [6]:
# 1) eliminar duplicados en orders
filas_antes = len(orders)
orders = orders.drop_duplicates().reset_index(drop=True)
print('Duplicados eliminados:', filas_antes - len(orders))
print('id_pedido repetidos restantes:', orders['id_pedido'].duplicated().sum())

# 2) convertir fechas a datetime
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], errors='coerce')
marketing['fecha'] = pd.to_datetime(marketing['fecha'], errors='coerce')

# (extra) verificar que ninguna fecha quedó inválida
print('\nFechas inválidas en orders:', orders['fecha_hora_pedido'].isna().sum())
print('Fechas inválidas en marketing:', marketing['fecha'].isna().sum())
print('\nRango orders:', orders['fecha_hora_pedido'].min().date(), '→', orders['fecha_hora_pedido'].max().date())
print('Rango marketing:', marketing['fecha'].min().date(), '→', marketing['fecha'].max().date())
print('\nTipos:', orders['fecha_hora_pedido'].dtype, '|', marketing['fecha'].dtype)

Duplicados eliminados: 100
id_pedido repetidos restantes: 0

Fechas inválidas en orders: 0
Fechas inválidas en marketing: 0

Rango orders: 2025-01-01 → 2025-06-30
Rango marketing: 2025-01-01 → 2025-06-29

Tipos: datetime64[ns] | datetime64[ns]


In [7]:
# 3) estandarizar textos en orders
# (extra) quitar espacios accidentales al inicio/final de todas las columnas de texto
cols_texto = ['pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto']
for col in cols_texto:
    orders[col] = orders[col].str.strip()

# unificar mayúsculas en país: 'mexico' → 'Mexico'
orders['pais'] = orders['pais'].str.title()

# unificar categoría: 'Electronica' → 'Electrónica' (igual que en catalog)
orders['categoria_producto'] = orders['categoria_producto'].replace('Electronica', 'Electrónica')

# verificación
print('pais:', orders['pais'].value_counts(dropna=False).to_dict())
print('\ncategoria orders :', sorted(orders['categoria_producto'].dropna().unique()))
print('categoria catalog:', sorted(catalog['categoria_producto'].unique()))
print('\n¿Categorías iguales en ambas tablas?',
      set(orders['categoria_producto'].dropna()) == set(catalog['categoria_producto']))

pais: {'Mexico': 8341, 'Colombia': 8304, 'Argentina': 8055, nan: 300}

categoria orders : ['Electrónica', 'Hogar', 'Moda']
categoria catalog: ['Electrónica', 'Hogar', 'Moda']

¿Categorías iguales en ambas tablas? True


In [8]:
# 4) recuperar nulos usando información de otras columnas/tablas

# 4a) categoria_producto desde catalog (cada producto tiene una sola categoría)
mapa_categoria = catalog.set_index('nombre_producto')['categoria_producto']
nulos_antes = orders['categoria_producto'].isna().sum()
orders['categoria_producto'] = orders['categoria_producto'].fillna(orders['nombre_producto'].map(mapa_categoria))
print(f'categoria_producto: {nulos_antes} nulos → {orders["categoria_producto"].isna().sum()}')

# 4b) pais desde otros pedidos del mismo usuario
mapa_pais = orders.dropna(subset=['pais']).groupby('id_usuario')['pais'].first()
nulos_antes = orders['pais'].isna().sum()
orders['pais'] = orders['pais'].fillna(orders['id_usuario'].map(mapa_pais))
print(f'pais: {nulos_antes} nulos → {orders["pais"].isna().sum()}')

# 4c) canal desde id_campaña en marketing ('paid_search_Mexico' → 'paid_search')
nulos_antes = marketing['canal'].isna().sum()
marketing['canal'] = marketing['canal'].fillna(marketing['id_campaña'].str.rsplit('_', n=1).str[0])
print(f'canal: {nulos_antes} nulos → {marketing["canal"].isna().sum()}')
print('canales:', marketing['canal'].unique())

# (extra) validar que cada producto tenga la misma categoría que en catalog
inconsistentes = (orders['categoria_producto'] != orders['nombre_producto'].map(mapa_categoria)) \
                 & orders['nombre_producto'].notna()
print('\nProductos con categoría distinta a catalog:', inconsistentes.sum())

categoria_producto: 80 nulos → 30
pais: 300 nulos → 19
canal: 101 nulos → 0
canales: ['organic' 'paid_search' 'social']

Productos con categoría distinta a catalog: 0


In [9]:
# 5) tratar nulos restantes
print('Nulos antes:'); print(orders.isna().sum()[orders.isna().sum() > 0])

# 5a) eliminar filas sin datos clave para calcular revenue y costo
cols_clave = ['nombre_producto', 'cantidad', 'precio_unitario', 'monto_descuento']
filas_antes = len(orders)
orders = orders.dropna(subset=cols_clave).reset_index(drop=True)
print(f'\nFilas eliminadas por datos clave faltantes: {filas_antes - len(orders)}')

# 5b) rellenar categóricas descriptivas (no afectan el cálculo de montos)
orders['pais'] = orders['pais'].fillna('Desconocido')
orders['dispositivo'] = orders['dispositivo'].fillna('unknown')
orders['fuente_referencia'] = orders['fuente_referencia'].fillna('unknown')

# verificación
print('\nNulos totales en orders:', orders.isna().sum().sum())
print('Nulos totales en marketing:', marketing.isna().sum().sum())
print('Filas actuales en orders:', len(orders))

Nulos antes:
pais                  19
dispositivo           20
fuente_referencia     30
nombre_producto       30
categoria_producto    30
cantidad              50
precio_unitario       50
monto_descuento       50
dtype: int64

Filas eliminadas por datos clave faltantes: 80

Nulos totales en orders: 0
Nulos totales en marketing: 0
Filas actuales en orders: 24920


In [10]:
# 6) eliminar valores inválidos y outliers
filas_antes = len(orders)

# 6a) cantidades y montos inválidos (≤ 0)
invalidos = (orders['cantidad'] <= 0) | (orders['monto_total'] <= 0)
print('Pedidos con cantidad o monto ≤ 0:', invalidos.sum())

# 6b) outliers de cantidad (errores de captura)
print('\nDistribución de cantidad:'); print(orders['cantidad'].value_counts().sort_index())
outliers = orders['cantidad'] > 100
print('\nPedidos con cantidad > 100:', outliers.sum())

orders = orders[~invalidos & ~outliers].reset_index(drop=True)
print(f'\nFilas eliminadas: {filas_antes - len(orders)} | Filas actuales: {len(orders)}')

# 6c) cantidad a entero
orders['cantidad'] = orders['cantidad'].astype(int)

# 6d) verificar consistencia: monto_total = cantidad × precio_unitario − monto_descuento
monto_calculado = orders['cantidad'] * orders['precio_unitario'] - orders['monto_descuento']
diferencia = (monto_calculado - orders['monto_total']).abs()
print('\nDiferencia máxima entre monto calculado y monto_total:', round(diferencia.max(), 2))
print('Pedidos con diferencia > 0.01:', (diferencia > 0.011).sum())

# (extra) resumen final de numéricas ya limpias
display(orders[['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']].describe().round(2))

Pedidos con cantidad o monto ≤ 0: 4

Distribución de cantidad:
-2.0            1
-1.0            3
 1.0        12330
 2.0        12576
 10000.0        6
 20000.0        4
Name: cantidad, dtype: int64

Pedidos con cantidad > 100: 10

Filas eliminadas: 14 | Filas actuales: 24906

Diferencia máxima entre monto calculado y monto_total: 0.01
Pedidos con diferencia > 0.01: 0


,cantidad,precio_unitario,monto_descuento,monto_total
count,24906.0,24906.00,24906.00,24906.00
mean,1.5,259.36,4.50,385.85
std,0.5,138.67,5.22,255.65
min,1.0,20.03,0.00,5.24
25%,1.0,138.52,0.00,180.62
50%,2.0,258.74,0.00,341.42
75%,2.0,380.36,10.00,517.62
max,2.0,499.96,15.00,999.89


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [11]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)
# (extra) verificar que los archivos se guardaron correctamente
for archivo in ['orders_clean.csv', 'catalog_clean.csv', 'marketing_clean.csv']:
    df_check = pd.read_csv(archivo)
    print(f'{archivo}: {df_check.shape[0]} filas, {df_check.shape[1]} columnas, {df_check.isna().sum().sum()} nulos')

orders_clean.csv: 24906 filas, 12 columnas, 0 nulos
catalog_clean.csv: 7 filas, 4 columnas, 0 nulos
marketing_clean.csv: 1620 filas, 5 columnas, 0 nulos


---
### ✅ Conclusión del Paso 1: ¿Podemos confiar en los datos?

**Sí, después de la limpieza.** Los datos originales tenían problemas que habrían distorsionado el análisis: con los outliers incluidos, el revenue se habría inflado ~5 veces.

**Problemas encontrados y acciones:**

| Problema | Acción | Filas afectadas |
|---|---|---|
| Filas duplicadas en `orders` | Eliminadas | 100 |
| Fechas en formato texto | Convertidas a `datetime` | todas |
| País con mayúsculas inconsistentes (`mexico`/`Mexico`) | Estandarizado con `str.title()` | ~2,487 |
| `Electronica` vs `Electrónica` entre tablas | Unificado según `catalog` | — |
| Nulos en `categoria_producto` | Recuperados desde `catalog` | 50 |
| Nulos en `pais` | Recuperados desde otros pedidos del mismo usuario | 281 |
| Nulos en `canal` (marketing) | Recuperados desde `id_campaña` | 101 |
| Sin producto, cantidad, precio o descuento | Eliminadas (no se pueden calcular montos) | 80 |
| País / dispositivo aún faltantes | Etiquetados como `Desconocido` / `unknown` | resto |
| Cantidades y montos ≤ 0 | Eliminados | 4 |
| Outliers de cantidad (10,000 y 20,000 uds.) | Eliminados como error de captura | 10 |

**Resultado:**
- `orders`: 25,100 → **24,906 filas** (se eliminó el 0.77%), 0 nulos.
- `catalog`: 7 filas, sin cambios de estructura.
- `marketing`: 1,620 filas, 0 nulos.
- `monto_total` es consistente con `cantidad × precio_unitario − monto_descuento` (diferencias ≤ 0.01 por redondeo).
- Periodo analizado: enero a junio de 2025 (marketing termina el 29/06 y los pedidos el 30/06).

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [12]:
# 2.1 Parte 1: rentabilidad del negocio

# unir orders con catalog para obtener el costo unitario de cada pedido
ventas = orders.merge(catalog[['nombre_producto', 'costo_unitario']],
                      on='nombre_producto', how='left', validate='many_to_one')

# (extra) verificar que el merge no perdió ni duplicó pedidos
print('Filas orders:', len(orders), '| Filas ventas:', len(ventas))
print('Pedidos sin costo asignado:', ventas['costo_unitario'].isna().sum())

# costo de cada pedido = cantidad × costo unitario
ventas['costo_total'] = ventas['cantidad'] * ventas['costo_unitario']

# KPIs
revenue = ventas['monto_total'].sum()
costo_productos = ventas['costo_total'].sum()
gasto_marketing = marketing['gasto'].sum()
profit_bruto = revenue - costo_productos
profit_neto = profit_bruto - gasto_marketing
margen_neto = profit_neto / revenue

print('\n=== RENTABILIDAD (ene–jun 2025) ===')
print(f'Revenue total:          ${revenue:>15,.2f}')
print(f'Costo de productos:     ${costo_productos:>15,.2f}')
print(f'Profit bruto:           ${profit_bruto:>15,.2f}')
print(f'Gasto en marketing:     ${gasto_marketing:>15,.2f}')
print(f'Profit neto:            ${profit_neto:>15,.2f}')
print(f'Margen neto:            {margen_neto:>16.1%}')


Filas orders: 24906 | Filas ventas: 24906
Pedidos sin costo asignado: 0

=== RENTABILIDAD (ene–jun 2025) ===
Revenue total:          $   9,610,018.94
Costo de productos:     $   3,828,869.01
Profit bruto:           $   5,781,149.93
Gasto en marketing:     $   2,871,843.53
Profit neto:            $   2,909,306.40
Margen neto:                       30.3%


In [13]:
# 2.1 Segmentos rentables: producto y categoría
ventas['profit'] = ventas['monto_total'] - ventas['costo_total']

def resumen_segmento(df, columna):
    """Agrupa por una columna y calcula revenue, costo, profit y margen."""
    tabla = (df.groupby(columna)
               .agg(pedidos=('id_pedido', 'count'),
                    unidades=('cantidad', 'sum'),
                    revenue=('monto_total', 'sum'),
                    costo=('costo_total', 'sum'),
                    profit=('profit', 'sum'))
               .sort_values('profit', ascending=False))
    tabla['margen_%'] = (tabla['profit'] / tabla['revenue'] * 100).round(1)
    return tabla.round(2)

print('=== RENTABILIDAD POR PRODUCTO ===')
display(resumen_segmento(ventas, 'nombre_producto'))

print('\n=== RENTABILIDAD POR CATEGORÍA ===')
display(resumen_segmento(ventas, 'categoria_producto'))

# (extra) precio promedio vs costo unitario por producto
comparacion = (ventas.groupby('nombre_producto')
                     .agg(precio_promedio=('precio_unitario', 'mean'),
                          costo_unitario=('costo_unitario', 'first'))
                     .round(2))
comparacion['ganancia_por_unidad'] = comparacion['precio_promedio'] - comparacion['costo_unitario']
print('\n=== PRECIO PROMEDIO vs COSTO UNITARIO ===')
display(comparacion.sort_values('ganancia_por_unidad'))

=== RENTABILIDAD POR PRODUCTO ===


,pedidos,unidades,revenue,costo,profit,margen_%
nombre_producto,,,,,,
Vacuum-Pro-Black,4170,6284,1621051.85,104314.40,1516737.45,93.6
Sneakers-Urban-42,4129,6172,1566075.93,106220.12,1459855.81,93.2
Phone-Pro-128GB,2733,4140,1057134.62,41896.80,1015237.82,96.0
Tablet-Standard-64GB,2764,4153,1062922.80,104697.13,958225.67,90.2
Blender-XL-Red,4176,6279,1611574.59,1109122.56,502452.03,31.2
Jacket-Winter-M,4166,6256,1606586.17,1184323.36,422262.81,26.3
Laptop-Gaming-16GB,2768,4198,1084672.98,1178294.64,-93621.66,-8.6



=== RENTABILIDAD POR CATEGORÍA ===


,pedidos,unidades,revenue,costo,profit,margen_%
categoria_producto,,,,,,
Hogar,8346,12563,3232626.44,1213436.96,2019189.48,62.5
Moda,8295,12428,3172662.10,1290543.48,1882118.62,59.3
Electrónica,8265,12491,3204730.40,1324888.57,1879841.83,58.7



=== PRECIO PROMEDIO vs COSTO UNITARIO ===


,precio_promedio,costo_unitario,ganancia_por_unidad
nombre_producto,,,
Laptop-Gaming-16GB,261.27,280.68,-19.41
Jacket-Winter-M,259.82,189.31,70.51
Blender-XL-Red,259.16,176.64,82.52
Tablet-Standard-64GB,257.53,25.21,232.32
Sneakers-Urban-42,257.23,17.21,240.02
Vacuum-Pro-Black,261.02,16.60,244.42
Phone-Pro-128GB,259.53,10.12,249.41


In [14]:
# 2.1 Segmentos: país, canal y dispositivo

# país y canal: se puede descontar el marketing porque el gasto está registrado por país y por canal
mkt_pais = marketing.groupby('pais')['gasto'].sum().rename('marketing')
mkt_canal = marketing.groupby('canal')['gasto'].sum().rename('marketing')

for col, mkt in [('pais', mkt_pais), ('fuente_referencia', mkt_canal)]:
    tabla = resumen_segmento(ventas, col).join(mkt)
    tabla['profit_neto'] = tabla['profit'] - tabla['marketing']
    tabla['ROI_mkt_%'] = (tabla['profit_neto'] / tabla['marketing'] * 100).round(1)
    print(f'\n=== RENTABILIDAD POR {col.upper()} ===')
    display(tabla.round(2))

# dispositivo: no hay gasto de marketing por dispositivo, solo profit bruto
print('\n=== RENTABILIDAD POR DISPOSITIVO ===')
display(resumen_segmento(ventas, 'dispositivo'))


=== RENTABILIDAD POR PAIS ===


,pedidos,unidades,revenue,costo,profit,margen_%,marketing,profit_neto,ROI_mkt_%
pais,,,,,,,,,
Mexico,8406,12666,3245422.63,1293130.39,1952292.24,60.2,988495.51,963796.73,97.5
Colombia,8359,12526,3200356.82,1280766.63,1919590.19,60.0,935653.42,983936.77,105.2
Argentina,8122,12265,3157426.04,1252119.34,1905306.70,60.3,947694.60,957612.10,101.0
Desconocido,19,25,6813.45,2852.65,3960.80,58.1,NaN,NaN,NaN



=== RENTABILIDAD POR FUENTE_REFERENCIA ===


,pedidos,unidades,revenue,costo,profit,margen_%,marketing,profit_neto,ROI_mkt_%
fuente_referencia,,,,,,,,,
paid_search,8257,12409,3209280.07,1240959.06,1968321.01,61.3,922374.20,1045946.81,113.4
social,8382,12609,3222510.32,1312735.79,1909774.53,59.3,976818.37,932956.16,95.5
organic,8267,12464,3178228.55,1275174.16,1903054.39,59.9,972650.96,930403.43,95.7



=== RENTABILIDAD POR DISPOSITIVO ===


,pedidos,unidades,revenue,costo,profit,margen_%
dispositivo,,,,,,
desktop,12661,19087,4901894.19,1961591.92,2940302.27,60.0
mobile,12225,18368,4701769.00,1865596.62,2836172.38,60.3
unknown,20,27,6355.75,1680.47,4675.28,73.6


In [15]:
# 2.1 Parte 2: comportamiento de ventas

# ticket promedio y productos por orden
ticket_promedio = ventas['monto_total'].mean()
productos_por_orden = ventas['cantidad'].mean()
print(f'Ticket promedio por orden:          ${ticket_promedio:,.2f}')
print(f'Cantidad promedio de productos/orden: {productos_por_orden:.2f}')

# producto más vendido (por unidades y por número de pedidos)
mas_vendido = (ventas.groupby('nombre_producto')
                     .agg(unidades=('cantidad', 'sum'), pedidos=('id_pedido', 'count'))
                     .sort_values('unidades', ascending=False))
print('\n=== PRODUCTOS MÁS VENDIDOS ===')
display(mas_vendido)
print(f'Producto más vendido (unidades): {mas_vendido.index[0]} con {mas_vendido["unidades"].iloc[0]:,} unidades')

# gasto de marketing por canal
gasto_canal = marketing.groupby('canal')['gasto'].sum().sort_values(ascending=False)
print('\n=== GASTO EN MARKETING POR CANAL ===')
display((pd.DataFrame({'gasto': gasto_canal,
                       '% del total': (gasto_canal / gasto_canal.sum() * 100).round(1)})).round(2))

Ticket promedio por orden:          $385.85
Cantidad promedio de productos/orden: 1.50

=== PRODUCTOS MÁS VENDIDOS ===


,unidades,pedidos
nombre_producto,,
Vacuum-Pro-Black,6284,4170
Blender-XL-Red,6279,4176
Jacket-Winter-M,6256,4166
Sneakers-Urban-42,6172,4129
Laptop-Gaming-16GB,4198,2768
Tablet-Standard-64GB,4153,2764
Phone-Pro-128GB,4140,2733


Producto más vendido (unidades): Vacuum-Pro-Black con 6,284 unidades

=== GASTO EN MARKETING POR CANAL ===


,gasto,% del total
canal,,
social,976818.37,34.0
organic,972650.96,33.9
paid_search,922374.20,32.1


---
### ✅ Conclusión del Paso 2: ¿El negocio es rentable?

**Sí. RappiPlus generó un profit neto de \$2.91 M en el semestre (margen neto de 30.3%).**

| KPI (ene–jun 2025) | Valor |
|---|---|
| Revenue total | \$9,610,018.94 |
| Costo de productos | \$3,828,869.01 |
| Profit bruto | \$5,781,149.93 (margen 60.2%) |
| Gasto en marketing | \$2,871,843.53 (29.9% del revenue) |
| **Profit neto** | **\$2,909,306.40 (margen 30.3%)** |
| Ticket promedio por orden | \$385.85 |
| Productos promedio por orden | 1.50 |
| Producto más vendido | Vacuum-Pro-Black (6,284 unidades) |

**💡 ¿Cómo se obtuvieron los costos por orden?**
`orders` no incluye el costo, así que se unió con `catalog` por `nombre_producto` (merge *left*, validado *many-to-one*) para traer el `costo_unitario`. Luego, **costo por orden = cantidad × costo_unitario**.

**Segmentos rentables y hallazgos:**
1. 🚨 **Laptop-Gaming-16GB pierde dinero:** su costo (\$280.68) supera su precio promedio de venta (\$261.27). Pérdida de \$19.41 por unidad y **−\$93,622** en el semestre (margen −8.6%). La categoría Electrónica oculta esta pérdida porque Phone y Tablet tienen márgenes de más de 90%.
2. ⭐ **Productos estrella:** Vacuum, Sneakers, Phone y Tablet tienen márgenes de 90–96%. Vacuum es el más vendido y el de mayor profit (\$1.52 M).
3. 📢 **paid_search es el canal más eficiente:** ROI de marketing de 113.4% frente a ~95% de social y organic, y aun así recibe la menor inversión (32.1%).
4. 🌎 **Países y dispositivos se comportan de forma similar** (márgenes de ~60%). Colombia tiene el mejor ROI (105.2%) y México el más bajo (97.5%).
5. 🔍 El canal *organic* registra ~\$973 K de gasto, un monto alto para un canal en teoría gratuito. Conviene revisarlo con marketing.

**Recomendaciones:**
- Revisar el precio de la Laptop-Gaming-16GB (subirlo por encima de \$280.68), negociar el costo con el proveedor o evaluar retirarla.
- Redistribuir presupuesto de social/organic hacia paid_search.
- Impulsar más unidades por pedido (combos o descuentos por volumen), ya que hoy el promedio es de solo 1.5.


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [16]:
import os
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión a la base de datos de TripleTen
# ======================
# Las credenciales NO se publican en el repositorio (buena práctica de seguridad).
# Para ejecutar los Pasos 3 y 4, define estas variables de entorno con los datos de acceso:
# DB_USER, DB_PASSWORD, DB_HOST, DB_PORT, DB_NAME
db_config = {
    'user': os.getenv('DB_USER'),
    'pwd': os.getenv('DB_PASSWORD'),
    'host': os.getenv('DB_HOST'),
    'port': os.getenv('DB_PORT', 5432),
    'db': os.getenv('DB_NAME')
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode': 'require'})

In [17]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [18]:
# Explorar eventos del funnel
print('Filas y columnas de events:', events.shape)
print('\nTipos de datos:'); print(events.dtypes)

# eventos disponibles: total de registros y usuarios únicos
query_eventos = '''
SELECT nombre_evento,
       COUNT(*)                   AS total_eventos,
       COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY usuarios_unicos DESC;
'''
display(pd.read_sql(query_eventos, con=engine))

# (extra) rango de fechas y nulos
print('\nRango de fechas:', events['timestamp_evento'].min(), '→', events['timestamp_evento'].max())
print('\nNulos por columna:'); print(events.isna().sum())

Filas y columnas de events: (120000, 8)

Tipos de datos:
id_usuario            object
id_sesion             object
nombre_evento         object
timestamp_evento      object
pais                  object
dispositivo           object
fuente_referencia     object
categoria_producto    object
dtype: object


,nombre_evento,total_eventos,usuarios_unicos
0,first_visit,29957,7796
1,add_to_cart,24157,7634
2,select_item,23887,7582
3,begin_checkout,17971,7208
4,add_payment_info,12018,6250
5,purchase,12010,6240



Rango de fechas: 2025-01-01 → 2025-06-30

Nulos por columna:
id_usuario            0
id_sesion             0
nombre_evento         0
timestamp_evento      0
pais                  0
dispositivo           0
fuente_referencia     0
categoria_producto    0
dtype: int64


In [19]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT
    CASE nombre_evento
        WHEN 'first_visit'      THEN 1
        WHEN 'select_item'      THEN 2
        WHEN 'add_to_cart'      THEN 3
        WHEN 'begin_checkout'   THEN 4
        WHEN 'add_payment_info' THEN 5
        WHEN 'purchase'         THEN 6
    END AS paso,
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY paso;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,paso,nombre_evento,usuarios_unicos
0,1,first_visit,7796
1,2,select_item,7582
2,3,add_to_cart,7634
3,4,begin_checkout,7208
4,5,add_payment_info,6250
5,6,purchase,6240


In [20]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel AS (
    SELECT
        CASE nombre_evento
            WHEN 'first_visit'      THEN 1
            WHEN 'select_item'      THEN 2
            WHEN 'add_to_cart'      THEN 3
            WHEN 'begin_checkout'   THEN 4
            WHEN 'add_payment_info' THEN 5
            WHEN 'purchase'         THEN 6
        END AS paso,
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos
    FROM events
    GROUP BY nombre_evento
)
SELECT
    paso,
    nombre_evento,
    usuarios_unicos,
    LAG(usuarios_unicos) OVER (ORDER BY paso) - usuarios_unicos              AS usuarios_perdidos,
    ROUND(usuarios_unicos * 100.0 / LAG(usuarios_unicos) OVER (ORDER BY paso), 2) AS conversion_paso_pct,
    ROUND(usuarios_unicos * 100.0 / FIRST_VALUE(usuarios_unicos) OVER (ORDER BY paso), 2) AS conversion_acumulada_pct
FROM funnel
ORDER BY paso;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,paso,nombre_evento,usuarios_unicos,usuarios_perdidos,conversion_paso_pct,conversion_acumulada_pct
0,1,first_visit,7796,NaN,NaN,100.00
1,2,select_item,7582,214.0,97.26,97.26
2,3,add_to_cart,7634,-52.0,100.69,97.92
3,4,begin_checkout,7208,426.0,94.42,92.46
4,5,add_payment_info,6250,958.0,86.71,80.17
5,6,purchase,6240,10.0,99.84,80.04


---
### ✅ Conclusión del Paso 3: ¿Dónde se pierden los usuarios?

**El mayor abandono ocurre al ingresar los datos de pago:** entre `begin_checkout` y `add_payment_info` solo avanza el **86.71%** de los usuarios, y se pierden **958 personas (13.3%)**. Es la mayor caída del funnel en número y en porcentaje.

| Paso | Evento | Usuarios únicos | Perdidos | Conversión vs paso anterior | Conversión acumulada |
|---|---|---|---|---|---|
| 1 | first_visit | 7,796 | — | — | 100.00% |
| 2 | select_item | 7,582 | 214 | 97.26% | 97.26% |
| 3 | add_to_cart | 7,634 | −52 | 100.69% | 97.92% |
| 4 | begin_checkout | 7,208 | 426 | 94.42% | 92.46% |
| 5 | **add_payment_info** | **6,250** | **958** | **86.71%** | **80.17%** |
| 6 | purchase | 6,240 | 10 | 99.84% | 80.04% |

**Hallazgos:**
1. 🎯 **Conversión final: 80.04%** (6,240 de 7,796 visitantes completaron una compra).
2. 🚨 **El cuello de botella está en el pago:** son usuarios con intención de compra (ya iniciaron el checkout) que abandonan antes de completar sus datos de pago.
3. ✅ **Una vez ingresado el pago, casi todos compran** (99.84%), así que el paso final no presenta problemas.
4. 🔀 **`add_to_cart` supera a `select_item` (100.69%):** 52 usuarios agregaron productos al carrito sin seleccionar un ítem antes (por ejemplo, desde listados o banners). El recorrido no siempre es lineal.

**Nota metodológica:** el funnel es agregado (usuarios únicos que realizaron cada evento en el periodo, sin exigir el orden). Una conversión de 80% es alta frente a e-commerce reales (2–5%), lo que es propio de estos datos.

**Recomendaciones:**
- Simplificar el formulario de pago (menos campos, autocompletar, guardar tarjetas).
- Ofrecer más métodos de pago locales y mostrar señales de seguridad.
- Mostrar el costo total (envío e impuestos) antes del checkout para evitar sorpresas.
- Validar los cambios con un test A/B antes de implementarlos (ver Paso 5).

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [21]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [22]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [23]:
# (extra) revisión de calidad: users y user_activity
for nombre, df in [('users', users), ('user_activity', user_activity)]:
    print(f'=== {nombre} ===')
    print('Filas y columnas:', df.shape)
    print('Tipos de datos:'); print(df.dtypes)
    print('Nulos:', df.isna().sum().sum(), '| Duplicados:', df.duplicated().sum())
    print()

print('Usuarios únicos en users:', users['id_usuario'].nunique())
print('Usuarios únicos en user_activity:', user_activity['id_usuario'].nunique())

# valores clave para definir las semanas de retención
print('\nValores de dias_despues_registro:')
print(user_activity['dias_despues_registro'].value_counts().sort_index())
print('\nValores de activo:')
print(user_activity['activo'].value_counts())

# rango de fechas de registro y planes
print('\nRango fecha_registro:', users['fecha_registro'].min(), '→', users['fecha_registro'].max())
print('\nTipos de plan:'); print(users['tipo_plan'].value_counts())

=== users ===
Filas y columnas: (8000, 5)
Tipos de datos:
id_usuario        object
fecha_registro    object
país              object
dispositivo       object
tipo_plan         object
dtype: object
Nulos: 0 | Duplicados: 0

=== user_activity ===
Filas y columnas: (32000, 4)
Tipos de datos:
id_usuario               object
fecha_actividad          object
dias_despues_registro     int64
activo                    int64
dtype: object
Nulos: 0 | Duplicados: 0

Usuarios únicos en users: 8000
Usuarios únicos en user_activity: 8000

Valores de dias_despues_registro:
7     8000
14    8000
21    8000
28    8000
Name: dias_despues_registro, dtype: int64

Valores de activo:
0    18685
1    13315
Name: activo, dtype: int64

Rango fecha_registro: 2025-01-01 → 2025-05-31

Tipos de plan:
free    5968
paid    2032
Name: tipo_plan, dtype: int64


In [24]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT
        id_usuario,
        CAST(DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS DATE) AS cohorte
    FROM users
),
retencion AS (
    SELECT
        c.cohorte,
        COUNT(DISTINCT c.id_usuario) AS clientes_iniciales,
        COUNT(DISTINCT CASE WHEN a.dias_despues_registro = 7  AND a.activo = 1 THEN a.id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE WHEN a.dias_despues_registro = 14 AND a.activo = 1 THEN a.id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE WHEN a.dias_despues_registro = 21 AND a.activo = 1 THEN a.id_usuario END) AS retenido_w3
    FROM cohortes c
    LEFT JOIN user_activity a ON c.id_usuario = a.id_usuario
    GROUP BY c.cohorte
)
SELECT
    cohorte,
    clientes_iniciales,
    retenido_w1,
    retenido_w2,
    retenido_w3,
    ROUND(retenido_w1 * 100.0 / clientes_iniciales, 2) AS semana_1,
    ROUND(retenido_w2 * 100.0 / clientes_iniciales, 2) AS semana_2,
    ROUND(retenido_w3 * 100.0 / clientes_iniciales, 2) AS semana_3
FROM retencion
ORDER BY cohorte;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,clientes_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01,1687,695,676,706,41.20,40.07,41.85


In [25]:
# (extra) Retención por tipo de plan
# ======================

query_retention_plan = '''
SELECT
    u.tipo_plan,
    COUNT(DISTINCT u.id_usuario) AS usuarios,
    ROUND(COUNT(DISTINCT CASE WHEN a.dias_despues_registro = 7  AND a.activo = 1 THEN a.id_usuario END) * 100.0
          / COUNT(DISTINCT u.id_usuario), 2) AS semana_1,
    ROUND(COUNT(DISTINCT CASE WHEN a.dias_despues_registro = 14 AND a.activo = 1 THEN a.id_usuario END) * 100.0
          / COUNT(DISTINCT u.id_usuario), 2) AS semana_2,
    ROUND(COUNT(DISTINCT CASE WHEN a.dias_despues_registro = 21 AND a.activo = 1 THEN a.id_usuario END) * 100.0
          / COUNT(DISTINCT u.id_usuario), 2) AS semana_3
FROM users u
LEFT JOIN user_activity a ON u.id_usuario = a.id_usuario
GROUP BY u.tipo_plan
ORDER BY u.tipo_plan;
'''

retencion_plan = pd.read_sql(query_retention_plan, con=engine)
retencion_plan

,tipo_plan,usuarios,semana_1,semana_2,semana_3
0,free,5968,35.05,35.82,35.66
1,paid,2032,62.40,59.89,60.14


---
### ✅ Conclusión del Paso 4: ¿Los usuarios regresan?

**Parcialmente. Alrededor del 42% de los usuarios se mantiene activo en cada una de las primeras 3 semanas; ~58% no regresa después de registrarse.**

| Cohorte | Clientes iniciales | Semana 1 | Semana 2 | Semana 3 |
|---|---|---|---|---|
| Enero 2025 | 1,627 | 42.84% | 41.06% | 40.32% |
| Febrero 2025 | 1,444 | 42.31% | 42.17% | 43.98% |
| Marzo 2025 | 1,636 | 41.38% | 43.09% | 42.18% |
| Abril 2025 | 1,606 | 42.34% | 43.40% | 41.28% |
| Mayo 2025 | 1,687 | 41.20% | 40.07% | 41.85% |

**Hallazgos:**
1. 📊 **La retención es estable (40–44%)** entre semanas y cohortes: no hay una caída progresiva ni cohortes que destaquen.
2. 🚪 **El abandono ocurre al inicio:** ~58% de los usuarios no está activo desde la primera semana. Quien se queda tiende a mantenerse.
3. 💳 **(Extra) Los usuarios del plan pagado regresan ~1.7 veces más:**

| Plan | Usuarios | Semana 1 | Semana 2 | Semana 3 |
|---|---|---|---|---|
| paid | 2,032 | 62.40% | 59.89% | 60.14% |
| free | 5,968 | 35.05% | 35.82% | 35.66% |

El promedio general (~42%) está cerca del plan *free* porque este representa el 75% de los usuarios.

**Notas metodológicas:**
- La retención mide si el usuario estuvo activo *en* la semana N (días 7, 14 y 21), no si estuvo activo en *todas* las semanas anteriores. Por eso un usuario puede faltar en la semana 1 y volver en la 3, y algunas cohortes suben ligeramente.
- La relación entre plan pagado y retención es una asociación, no necesariamente causal: los usuarios más activos podrían ser quienes eligen pagar.
- En la tabla `users`, la columna de país se llama `país` (con tilde), a diferencia de la documentación y de la tabla `events` (`pais`).

**Recomendaciones:**
- Fortalecer el *onboarding* de la primera semana (beneficio de bienvenida, recordatorios en los días 1–3, incentivo para la segunda compra).
- Impulsar la conversión de *free* a *paid* (por ejemplo, una prueba gratuita del plan pagado) y validar su impacto con un test A/B.

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
### Hipótesis estadística

**Métrica principal:** tasa de conversión de compra (columna `convirtio`: 1 = completó la compra, 0 = no). *Nota: el enunciado la llama `conversion`, pero en el dataset se llama `convirtio`.*

   - **H₀ (Hipótesis nula):** la tasa de conversión del grupo **tratamiento** (nueva UI del checkout) es **igual** a la del grupo **control** (UI actual). La diferencia observada se debe al azar.  
     *p_tratamiento = p_control*
   - **H₁ (Hipótesis alternativa):** la tasa de conversión del grupo **tratamiento** es **diferente** a la del grupo **control**.  
     *p_tratamiento ≠ p_control*

**Test estadístico:** prueba Z para dos proporciones (bilateral).  
**Nivel de significancia alpha:** 0.05

In [26]:
# Paso 5: cargar y explorar el experimento
experiment = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')

print('Filas y columnas:', experiment.shape)
display(experiment.head())
print('\nTipos de datos:'); print(experiment.dtypes)
print('\nNulos:', experiment.isna().sum().sum(), '| Duplicados:', experiment.duplicated().sum())

# tamaño de cada grupo y tasa de conversión por variante
print('\n=== USUARIOS Y CONVERSIÓN POR VARIANTE ===')
resumen_ab = experiment.groupby('variante')['convirtio'].agg(usuarios='count', conversiones='sum', tasa_conversion='mean')
resumen_ab['tasa_conversion_%'] = (resumen_ab['tasa_conversion'] * 100).round(2)
display(resumen_ab.drop(columns='tasa_conversion'))

# (extra) verificar que ningún usuario esté en ambos grupos
print('Usuarios en ambos grupos:', experiment.groupby('id_usuario')['variante'].nunique().gt(1).sum())

Filas y columnas: (10000, 7)


,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12



Tipos de datos:
id_usuario          object
variante            object
convirtio            int64
dispositivo         object
pais                object
duracion_sesion    float64
timestamp           object
dtype: object

Nulos: 0 | Duplicados: 0

=== USUARIOS Y CONVERSIÓN POR VARIANTE ===


,usuarios,conversiones,tasa_conversion_%
variante,,,
control,4965,779,15.69
tratamiento,5035,820,16.29


Usuarios en ambos grupos: 0


In [27]:
# Paso 5: prueba Z para dos proporciones
alpha = 0.05

# datos de cada grupo
conv_trat = experiment.loc[experiment['variante'] == 'tratamiento', 'convirtio']
conv_ctrl = experiment.loc[experiment['variante'] == 'control', 'convirtio']

x_trat, n_trat = conv_trat.sum(), conv_trat.count()
x_ctrl, n_ctrl = conv_ctrl.sum(), conv_ctrl.count()
p_trat, p_ctrl = x_trat / n_trat, x_ctrl / n_ctrl

# proporción combinada (bajo H0 ambos grupos tienen la misma tasa)
p_comb = (x_trat + x_ctrl) / (n_trat + n_ctrl)

# estadístico Z y p-valor bilateral
error_std = np.sqrt(p_comb * (1 - p_comb) * (1 / n_trat + 1 / n_ctrl))
z = (p_trat - p_ctrl) / error_std
p_valor = 2 * stats.norm.sf(abs(z))

print(f'Conversión tratamiento: {p_trat:.2%} ({x_trat} de {n_trat})')
print(f'Conversión control:     {p_ctrl:.2%} ({x_ctrl} de {n_ctrl})')
print(f'Diferencia absoluta:    {(p_trat - p_ctrl) * 100:+.2f} puntos porcentuales')
print(f'Lift relativo:          {(p_trat - p_ctrl) / p_ctrl:+.2%}')
print(f'\nEstadístico Z: {z:.4f}')
print(f'p-valor:       {p_valor:.4f}')

# (extra) intervalo de confianza del 95% para la diferencia
error_dif = np.sqrt(p_trat * (1 - p_trat) / n_trat + p_ctrl * (1 - p_ctrl) / n_ctrl)
z_crit = stats.norm.ppf(1 - alpha / 2)
ic_inf = (p_trat - p_ctrl) - z_crit * error_dif
ic_sup = (p_trat - p_ctrl) + z_crit * error_dif
print(f'IC 95% de la diferencia: [{ic_inf * 100:+.2f}, {ic_sup * 100:+.2f}] puntos porcentuales')

# decisión
if p_valor < alpha:
    print('\n✅ Se rechaza H0: la nueva UI tiene un efecto estadísticamente significativo en la conversión.')
else:
    print('\n❌ No se rechaza H0: no hay evidencia suficiente de que la nueva UI cambie la conversión.')

Conversión tratamiento: 16.29% (820 de 5035)
Conversión control:     15.69% (779 de 4965)
Diferencia absoluta:    +0.60 puntos porcentuales
Lift relativo:          +3.80%

Estadístico Z: 0.8133
p-valor:       0.4161
IC 95% de la diferencia: [-0.84, +2.03] puntos porcentuales

❌ No se rechaza H0: no hay evidencia suficiente de que la nueva UI cambie la conversión.


In [28]:
# (extra) Validaciones del experimento

# 1) SRM (Sample Ratio Mismatch): ¿la división 50/50 entre grupos fue correcta?
tamanos = experiment['variante'].value_counts()
chi2_srm, p_srm = stats.chisquare(tamanos)
print('=== 1) Balance de grupos (SRM) ===')
print(tamanos.to_dict())
print(f'p-valor SRM: {p_srm:.4f} →', 'grupos balanceados ✅' if p_srm >= alpha else 'posible problema de asignación ⚠️')

# 2) efecto por dispositivo (análisis exploratorio)
print('\n=== 2) Conversión por dispositivo y variante ===')
for disp, grupo in experiment.groupby('dispositivo'):
    t = grupo.loc[grupo['variante'] == 'tratamiento', 'convirtio']
    c = grupo.loc[grupo['variante'] == 'control', 'convirtio']
    p_c = (t.sum() + c.sum()) / (t.count() + c.count())
    z_d = (t.mean() - c.mean()) / np.sqrt(p_c * (1 - p_c) * (1 / t.count() + 1 / c.count()))
    p_d = 2 * stats.norm.sf(abs(z_d))
    print(f'{disp:8s} | control: {c.mean():.2%} | tratamiento: {t.mean():.2%} | '
          f'dif: {(t.mean() - c.mean()) * 100:+.2f} pp | p-valor: {p_d:.4f}')

=== 1) Balance de grupos (SRM) ===
{'tratamiento': 5035, 'control': 4965}
p-valor SRM: 0.4839 → grupos balanceados ✅

=== 2) Conversión por dispositivo y variante ===
desktop  | control: 18.38% | tratamiento: 18.19% | dif: -0.19 pp | p-valor: 0.8625
mobile   | control: 12.94% | tratamiento: 14.35% | dif: +1.41 pp | p-valor: 0.1478


---
### ✅ Conclusión del Paso 5: ¿Los cambios funcionan?

**No hay evidencia suficiente de que la nueva UI del checkout mejore la conversión.** No se rechaza H₀ (p-valor = 0.416 > α = 0.05).

| Grupo | Usuarios | Conversiones | Tasa de conversión |
|---|---|---|---|
| Control (UI actual) | 4,965 | 779 | 15.69% |
| Tratamiento (nueva UI) | 5,035 | 820 | 16.29% |

| Resultado del test | Valor |
|---|---|
| Diferencia absoluta | +0.60 puntos porcentuales (lift relativo +3.8%) |
| Estadístico Z | 0.81 |
| p-valor | 0.416 |
| IC 95% de la diferencia | [−0.84, +2.03] pp (incluye el 0) |

**Interpretación:**
1. 📉 La diferencia de +0.60 pp **puede deberse al azar**: el intervalo de confianza incluye el 0, así que el efecto real podría ser nulo o incluso negativo.
2. ⚖️ **No rechazar H₀ no prueba que la UI no funcione:** indica que, con este tamaño de muestra, no se detecta un efecto. Un efecto menor a ~1 pp requeriría una muestra mayor.

**Validaciones (extra):**
- ✅ **Balance de grupos (SRM):** p-valor = 0.484, así que la asignación 50/50 fue correcta y el experimento es confiable.
- 📱 **Por dispositivo (exploratorio):** en *desktop* no hay efecto (−0.19 pp, p = 0.86); en *mobile* hay una mejora de +1.41 pp (p = 0.15), no significativa. Es una señal a investigar, no una conclusión, porque los análisis por subgrupos aumentan el riesgo de falsos positivos (comparaciones múltiples).
- 📱 Independientemente de la variante, **mobile convierte menos que desktop** (~13–14% vs ~18%).

**Recomendaciones:**
- **No implementar la nueva UI** solo con base en este experimento: no muestra un impacto significativo.
- Diseñar un **nuevo experimento enfocado en mobile** y en el **formulario de pago**, que es la etapa con mayor abandono según el funnel (Paso 3), con una muestra más grande para detectar efectos pequeños.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---
### ✅ Paso 6: Dashboard en Power BI

El dashboard **`RappiPlus_Dashboard.pbix`** está en la carpeta del proyecto, junto a los 3 CSV limpios.

- **Fuentes:** `orders_clean.csv` y `catalog_clean.csv` (la tabla de marketing no se requiere en este paso).
- **Modelo estrella:** `orders_clean` (hechos) relacionado 1:* con `catalog_clean` (por producto) y con `dim_fecha` (tabla calendario marcada como tabla de fechas), sin duplicados.
- **Página 1 – Overview Ejecutivo:** KPIs (Revenue, Profit, Margen %, Ticket promedio, Productos por orden, Pedidos), evolución mensual, acumulado YTD y Revenue/Profit por producto.
- **Página 2 – Detalle de pedidos (drill-through):** tabla de órdenes con profit en rojo (negativo) y verde (positivo), unidades vendidas por producto y filtros por fecha, categoría y país.
- **Filtros sincronizados** de Categoría y País entre ambas páginas.
- **Nota:** en el dashboard, *Profit = Revenue − costo de productos* (profit bruto, USD 5.78 M). El profit neto con marketing (USD 2.91 M) está calculado en el Paso 2.

---
## 🧭 Conclusión general

| Pregunta del negocio | Respuesta |
|---|---|
| **1. ¿Podemos confiar en los datos?** | Sí, tras la limpieza: se eliminaron duplicados, inválidos y outliers (25,100 → 24,906 pedidos, −0.77%) y se recuperaron 432 datos faltantes desde otras columnas. |
| **2. ¿El negocio es rentable?** | Sí: Revenue de USD 9.61 M, profit bruto de USD 5.78 M (60.2%) y profit neto de USD 2.91 M (30.3%). **Laptop-Gaming-16GB pierde dinero** (−USD 93.6 K): se vende por debajo de su costo. |
| **3. ¿Dónde se pierden los usuarios?** | En el **pago**: de `begin_checkout` a `add_payment_info` solo avanza el 86.71% (958 usuarios perdidos). Conversión final: 80.04%. |
| **4. ¿Los usuarios regresan?** | Parcialmente: ≈42% sigue activo en las semanas 1–3. Los usuarios **paid** retienen ≈1.7 veces más que los **free** (≈60% vs ≈35%). |
| **5. ¿Los cambios funcionan?** | **No hay evidencia** de que la nueva UI del checkout mejore la conversión (p = 0.416). En *mobile* se observa una mejora de +1.41 pp, no significativa (p = 0.15). |
| **6. ¿Cómo lo comunicamos?** | Con un dashboard ejecutivo de 2 páginas en Power BI, con drill-through del KPI general al detalle de cada pedido. |

### 🎯 Recomendaciones (por prioridad)
1. **Corregir el precio de Laptop-Gaming-16GB** (por encima de USD 280.68), renegociar su costo con el proveedor o evaluar retirarla.
2. **Simplificar el paso de pago**, especialmente en *mobile* (convierte ≈13–14% vs ≈18% en desktop), y validarlo con un nuevo test A/B enfocado en mobile y con una muestra mayor.
3. **Reforzar el onboarding de la primera semana** e impulsar la conversión de *free* a *paid*, validando su impacto con un experimento.
4. **Redistribuir el presupuesto de marketing hacia `paid_search`** (ROI de 113% vs ≈95% en otros canales) y revisar el gasto del canal *organic*.
5. **Aumentar las unidades por pedido** (hoy 1.5) con combos o descuentos por volumen.

### ⚠️ Limitaciones
- Datos de solo 6 meses (ene–jun 2025): no permiten comparaciones interanuales (YoY).
- Algunas métricas son atípicas frente a e-commerce reales (conversión de 80%), propio de datos simulados.
- El funnel es agregado (no secuencial por usuario) y la relación plan pagado–retención es una asociación, no necesariamente causal.

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

### 📎 Entrega del Dashboard

Subí mi proyecto a la plataforma: el dashboard se entrega como **archivo**, en la misma carpeta de este notebook:

| Archivo | Contenido |
|---|---|
| `RappiPlus_Dashboard.pbix` | Dashboard de Power BI (2 páginas: Overview Ejecutivo y Detalle de pedidos) |
| `orders_clean.csv` | Pedidos limpios (Paso 1) |
| `catalog_clean.csv` | Catálogo de productos limpio (Paso 1) |
| `marketing_clean.csv` | Gasto de marketing limpio (Paso 1) |

**Para revisarlo:** descargar `RappiPlus_Dashboard.pbix` y abrirlo en Power BI Desktop. El archivo incluye los datos importados, por lo que se puede explorar completo (filtros, drill-through y medidas) sin necesidad de actualizar la fuente.

**Drill-through:** en la página *Overview Ejecutivo*, clic derecho sobre un producto del gráfico *Revenue y Profit por producto* → **Drill through** → **Detalle de pedidos**.